# Script Cue Building with SQLite Database Integration (SQLModel)

This notebook parses Fountain screenplay format and generates character mute/unmute cues, storing them in a SQLite database using SQLModel ORM.

In [1]:
from fountain import fountain
from qlab.database_sqlmodel import CueDatabase
from pathlib import Path
import re
# from rich import print

## Load and Parse Script

In [2]:
with open('../seussical/scripts/seussical.fountain', 'r') as file:
    f = fountain.Fountain(file.read())
    f.parse()

print(f"Loaded {len(f.elements)} script elements")

Loaded 6125 script elements


## Helper Function: Predict Future Speaking

In [3]:
def split_characters(characters: str) -> list[str]:
    """Clean and split character headings"""
    # Remove parenthesis
    characters = re.sub(r'\([^)]*\)', '', characters)
    # split the characters by '&'
    return characters.split(' & ')

def speaks_within(book, character, n: int = 7):
    """Check if character speaks within next n dialogue blocks or before scene change.
    
    Args:
        book: List of script elements to search
        character: Character name to look for
        n: Number of dialogue blocks to look ahead
    
    Returns:
        True if character speaks within window, False otherwise
    """
    dialogues = 0
    for i, element in enumerate(book):
        if dialogues >= n:
            return False
        if element.element_type == 'Scene Heading':
            return False
        if element.element_type == 'Character':
            characters = split_characters(element.element_text)
            if character in characters:
                return True
            dialogues += 1
    return False

In [4]:
# Generate the character list
characters = set()
for element in f.elements:
    if element.element_type == 'Character':
        chars = split_characters(element.element_text)
        for char in chars:
            characters.add(char.strip())
characters = sorted(list(characters))
print(f"Found {len(characters)} unique characters:\n")
# for c in characters:
#     print(c.title())

Found 64 unique characters:



## Generate Cues from Script

In [5]:
from qlab.models import Cue
import re
script = f.elements

cues = []
active = set()
page = 1

for i, element in enumerate(script):
    # Get the page number
    if element.element_type == 'Comment':
        if re.match(r'^Page \d+$', element.element_text):
            page = int(re.search(r'\d+', element.element_text).group())
            # print(f"Page changed to {page}")
            continue
    # For every Character element
    if element.element_type == 'Character':
        # Get preview of dialogue line
        line_start = script[i+1].element_text[:40] + '...' if i+1 < len(script) else ''
        line_end = '...' + script[i+1].element_text[-40:] if i+1 < len(script) else ''
  
        # Get character name(s)
        characters = split_characters(element.element_text)
        mutes, unmutes = [], []
        for character in characters:
            # If they aren't active, unmute
            if character not in active:        
                unmutes.append(character)
                active.add(character)
            
            # If they won't speak again soon, mute them
            if not speaks_within(script[i+1:], character):
                cue_name = f'(p{page}) mute {character.title()}: "{line_end}"'
                # print(f"{cue_name}: {line_end}")
                mutes.append(character)
                active.remove(character)
        if unmutes:
            cues.append(
                Cue(
                    name=f'(p{page}) unmute {", ".join([c.title() for c in unmutes])}: "{line_start}"',
                    number=len(cues)+1,
                    point=0
                )
            )
        if mutes:
            cues.append(
                Cue(
                    name=f'(p{page}) mute {", ".join([c.title() for c in mutes])}: "{line_end}"',
                    number=len(cues)+1,
                    point=0
                )
            )

print(f"\nGenerated {len(cues)} total cues")
cues[:10]


Generated 1236 total cues


[Cue(number=1, point=0, name='(p1) unmute Jojo : "Now that is a very unusual hat.\nI wonder..."', dca01Channels=None, dca02Channels=None, dca03Channels=None, dca04Channels=None, dca05Channels=None, dca06Channels=None, dca07Channels=None, dca08Channels=None, dca09Channels=None, dca10Channels=None, dca11Channels=None, dca12Channels=None, dca01Label=None, dca02Label=None, dca03Label=None, dca04Label=None, dca05Label=None, dca06Label=None, dca07Label=None, dca08Label=None, dca09Label=None, dca10Label=None, dca11Label=None, dca12Label=None, channelPositions=None, channelProfiles=None, fxMutes=None, channelFX=None, snippets=None, qLabCue=None, channelLevels=None, scenes=None, colour=None, scenePoints=None),
 Cue(number=2, point=0, name='(p1) mute Jojo : "... a sort of a kind of a hat-wearing…\nCat!"', dca01Channels=None, dca02Channels=None, dca03Channels=None, dca04Channels=None, dca05Channels=None, dca06Channels=None, dca07Channels=None, dca08Channels=None, dca09Channels=None, dca10Channels

## Save to CSV (Legacy Export)

In [6]:
# from csv import writer

# with open('cues.csv', 'w') as file:
#     csv = writer(file)
#     csv.writerow(['cue_type', 'character', 'line'])
#     for cue in cues:
#         csv.writerow(cue)

# print(f"Saved {len(cues)} cues to cues.csv")

## Save to SQLite Database

Create or use an existing database to store the cues.

In [7]:
# Choose database - either create new or use existing
db_path = 'mix/seuss5.tmix'  # Change to use existing database

# You can also use the example database:
# db_path = 'mix/SheKillsMonsters.sqlite'

print(f"Using database: {db_path}")


Using database: mix/seuss5.tmix


In [8]:
from sqlmodel import create_engine, SQLModel, Session, select
from qlab import models
from rich import print

engine = create_engine(f'sqlite:///{db_path}') #, echo=True)
SQLModel.metadata.create_all(engine)

In [9]:
with Session(engine) as session:
    session.add_all(cues)
    session.commit()

## Verify Database Contents

In [10]:
with Session(engine) as session:
    cues = select(models.Cue)
    result = session.exec(cues).all()
    print(f"Total cues in database: {len(result)}")
    print("\nLast 5 cues:")
    for cue in result[-5:]:
        print(f"  {cue.number}.{cue.point}: {cue.name}")

Total cues in database: 1236

Last 5 cues:

1232.0: (p101) mute Group 1, Group 2, Group 3, Group 4: "...SEUSS!
SEUSS! SEUSS
SEUSS!
SEUSS! SEUSS!"

1233.0: (p101) mute Jojo: "...Seuss!"

1234.0: (p101) mute All: "... RAIN?
COULD YOU? WOULD YOU? ON A TRAIN?"

1235.0: (p102) unmute All : "NOT WITH A GOAT. NOT ON A BOAT.
NOT IN T..."

1236.0: (p102) mute All : "...S AND HAM!
I DO NOT LIKE THEM, SAM-I-AM!"